# Training a Basic Classifier on Clidemia Hirta Leaves
This notebook should give us a classification baseline on our dataset 

In [6]:
import os
import tempfile

# Set tempfile
tempfile.tempdir = "/local/scratch/carlyn.1/tmp"

# Setting GPU
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

## Training Loop

In [7]:
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import SGD
from torch.nn import BCELoss
import torchvision.transforms as T

from inv_plts.data.datasets import BasicInvasivePlantsDataset
from inv_plts.models.simple import BasicInvasiveSpeciesCNN

print("Loading Model")
# net = BasicInvasiveSpeciesCNN().cuda()


class BasicInvasiveSpeciesCNNv2(nn.Module):
    def __init__(
        self,
        in_dims=3,
        layer_feature_dims=[16, 32, 64, 128, 256],
        layer_widths=[3, 3, 3, 3, 3],
    ):
        super().__init__()
        self.sigmoid = nn.Sigmoid()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.layers = []
        for i, (feat_dims, width) in enumerate(zip(layer_feature_dims, layer_widths)):
            if i == 0:
                in_dim = in_dims
            else:
                in_dim = layer_feature_dims[i - 1]

            self.layers.append(
                self._construct_layer(
                    in_dim=in_dim, out_dim=feat_dims, layer_width=width
                )
            )

        self.layer = nn.ModuleList(self.layers)
        self.fc = nn.Linear(in_features=256, out_features=5, bias=True)

    def _construct_layer(self, in_dim, out_dim, layer_width):
        dims = [(in_dim, out_dim)] + [
            (out_dim, out_dim) for _ in range(layer_width - 1)
        ]
        inner_layers = [
            nn.Sequential(
                nn.Conv2d(in_d, out_d, kernel_size=3), nn.BatchNorm2d(out_d), nn.ReLU()
            )
            for in_d, out_d in dims
        ]

        return nn.Sequential(*inner_layers, nn.MaxPool2d(kernel_size=2))

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer(h)

        feats = torch.flatten(self.avgpool(h), start_dim=1)
        out = self.sigmoid(self.fc(feats))
        return out


net = BasicInvasiveSpeciesCNNv2().cuda()

# image_root_path = Path("/local/scratch/carlyn.1/invasive-image-sessions")
image_root_path = Path("img_tmp")
metadata_csv = Path("../tmp/metadata/linked_metadata.csv")

transforms = T.Compose(
    [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

print("Loading Dataset")
dataset = BasicInvasivePlantsDataset(
    image_root=image_root_path, metadata_csv=metadata_csv, transform=transforms
)
data: BasicInvasivePlantsDataset.DataStructure
optimizer = SGD(net.parameters(), lr=0.003)
loss_fn = BCELoss()


@dataclass
class BatchStructure:
    images: torch.Tensor
    labels: torch.Tensor


def collate_fn(batch):
    images = []
    labels = []
    for item in batch:
        images.append(item.image)
        labels.append(item.label)

    return BatchStructure(images=torch.stack(images), labels=torch.stack(labels))


dataloader = DataLoader(
    dataset, batch_size=8, num_workers=4, shuffle=False, collate_fn=collate_fn
)

Loading Model
Loading Dataset


In [8]:
# dataset.preprocess("img_tmp")

In [9]:
import plotly.graph_objects as go
import numpy as np

losses = []
accuracies = []


class LossGraph:
    def __init__(self):
        self.fig = go.FigureWidget(
            [
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name="Prediction Loss",
                )
            ]
        )

    def update(self):
        self.fig.data[0].x = list(range(len(losses)))
        self.fig.data[0].y = losses

    def display(self):
        display(self.fig)


class AccuracyGraph:
    def __init__(self):
        class_names = ["Healthly", "Leaf Miner", "Rust", "Other Insect", "Mechanical"]
        self.fig = go.FigureWidget(
            [
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines+markers",
                    name=f"Accuracy {class_names[i]}",
                )
                for i in range(5)
            ]
        )

    def update(self):
        for cls_idx in range(5):
            self.fig.data[cls_idx].x = list(range(len(accuracies)))
            self.fig.data[cls_idx].y = np.array(accuracies)[:, cls_idx]

    def display(self):
        display(self.fig)


loss_graph = LossGraph()
loss_graph.display()

accuracy_graph = AccuracyGraph()
accuracy_graph.display()

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Prediction Loss',
              'type': 'scatter',
              'uid': 'c389bab7-8e9c-4b61-a6c7-7aec488fcdf6',
              'x': [],
              'y': []}],
    'layout': {'template': '...'}
})

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Accuracy Healthly',
              'type': 'scatter',
              'uid': '3e7d6bf3-cb51-4ad7-be11-33d2e10a0a6f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Leaf Miner',
              'type': 'scatter',
              'uid': '7a968a64-ad18-465b-80c4-5041350f2781',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Rust',
              'type': 'scatter',
              'uid': '9daf260d-42a8-4cac-883c-59dba57028b8',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Other Insect',
              'type': 'scatter',
              'uid': 'e920a4ab-67cf-43c2-972b-d7519ba873ec',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Accuracy Mechanical',
       

In [10]:
from tqdm import tqdm

losses = []
accuracies = []
NUM_EPOCHS = 100
tbar = tqdm(range(NUM_EPOCHS), desc="Epoch Training", position=1, leave=True)
for epoch in tbar:
    total = 0
    total_loss = 0
    total_correct = []
    for data in tqdm(dataloader, desc="Batch Training", position=0, leave=False):
        # Training
        optimizer.zero_grad()
        out = net(data.images.cuda())
        loss = loss_fn(out, data.labels.cuda())
        loss.backward()
        optimizer.step()

        damage_predicted = (out >= 0.5).detach().cpu().type(torch.LongTensor)
        correct = damage_predicted == data.labels.detach().cpu().type(torch.LongTensor)

        # Tracking
        total += len(data.images)
        total_loss += loss.item()
        total_correct.append(correct.sum(0).numpy())

    # Tracking
    losses.append(total_loss)
    accuracy = np.stack(total_correct).sum(0) / total
    accuracies.append(accuracy)
    tbar.set_postfix({"loss": total_loss})

    loss_graph.update()
    accuracy_graph.update()

Epoch Training: 100%|██████████| 100/100 [02:36<00:00,  1.56s/it, loss=1.1]
